# 03 — Main comparison grid (Prompt 12 plan → Prompt 13 evaluation)

Runs every arm of protocol v1.1. **Starts only if the baseline was validated** (Prompt 10 Part B filled in
and you created `results/baseline/VALIDATED`). Kaggle: GPU T4 x2, Internet on, Persistence Files.
Runs are resumable: if a session ends, rerun the notebook and finished items are skipped.
Order = most important first, so a quota cut loses the least important runs.

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("orchestrating-the-virtual-tumor-board"):
    subprocess.run(["git","clone","--depth","1","https://github.com/vibhum-phd/orchestrating-the-virtual-tumor-board.git"], check=True)
os.chdir("/kaggle/working/orchestrating-the-virtual-tumor-board/board-eval")
!pip install -q -r requirements.txt -r requirements-gpu.txt
COMMIT = subprocess.run(["git","rev-parse","HEAD"],capture_output=True,text=True).stdout.strip()
print("commit", COMMIT); open("results/COMMIT_USED.txt","w").write(COMMIT + "\n")

## Gate: baseline validated? leakage gate passed?

In [ ]:
import json, os, sys
assert os.path.exists("results/baseline/VALIDATED"), "STOP: complete Prompt 10 Part B and create results/baseline/VALIDATED first"
for f in ["results/env/leak_pilot_vs_test.json", "results/env/leak_medmcqa_vs_medqa.json"]:
    if os.path.exists(f):
        d = json.load(open(f)); assert d["n_flagged_pairs"] == 0, f"STOP: leakage gate failed in {f}"
    else:
        print("WARNING: missing", f, "- rerun notebook 01 cell 7")
print("gates passed")

## 1. Primary set, seed 0 (all four arms) — the runs the primary analysis needs

In [ ]:
for ds in ["medqa_onc", "medmcqa_onc"]:
    for m in ["cot", "sc", "board_roles", "board_generic"]:
        !python scripts/run_experiment.py --dataset {ds} --method {m} --seed 0 --backend vllm --chunk 32

## 2. Secondary set: full MedQA test, seed 0 (cut first if quota runs short)

In [ ]:
for m in ["cot", "sc", "board_roles", "board_generic"]:
    !python scripts/run_experiment.py --dataset medqa_all --method {m} --seed 0 --backend vllm --chunk 32

## 3. Seed sensitivity: primary set, seeds 1 and 2 (cut second)

In [ ]:
for seed in [1, 2]:
    for ds in ["medqa_onc", "medmcqa_onc"]:
        for m in ["cot", "sc", "board_roles", "board_generic"]:
            !python scripts/run_experiment.py --dataset {ds} --method {m} --seed {seed} --backend vllm --chunk 32

## 4. Evaluate (integrity checks run first; the script stops on any FAIL)

In [ ]:
!python scripts/evaluate.py --raw results/raw --out results
!cat results/integrity_report.json | head -60

In [ ]:
!cd /kaggle/working/orchestrating-the-virtual-tumor-board/board-eval && zip -qr /kaggle/working/main_grid_outputs.zip results && ls -la /kaggle/working/main_grid_outputs.zip

## STOP — share `main_grid_outputs.zip`
Interpretation (Prompt 14) and the paper (Prompt 15) are written only from these files.